### Robert Gravatt - Data 201 Homework 2

This Jupyter Notebook file was created using PyCharm.

Part A – Core Wrangling (Method Chaining)
The following code uses a single chained expression to filter, transform, group, and aggregate the housing data.

In [4]:
import pandas as pd

# Load the dataset
df = pd.read_csv("housing.csv")

# Core Wrangling Chained Expression
summary_table = (
    df.loc[(df["price"] > 250000) & (df["size"] > 1000)]
    .assign(price_per_sqft = lambda x: x["price"] / x["size"])
    .groupby("neighborhood")
    .agg(
        mean_pps = ("price_per_sqft", "mean"),
        median_pps = ("price_per_sqft", "median"),
        home_count = ("price_per_sqft", "count")
    )
    .sort_values("mean_pps", ascending=False)
)

print(summary_table)

                mean_pps   median_pps  home_count
neighborhood                                     
Downtown      977.820905  1001.557049          99
Midtown       921.141446   901.992377          92
Suburb        861.917078   836.878644         157
Uptown        860.935608   843.653468          99
Waterfront    849.508891   792.663291          48


Part B – Translation to dplyr

library(dplyr)

summary_table <- df |>

  filter(price > 250000, size > 1000) |>

  mutate(price_per_sqft = price / size) |>
  group_by(neighborhood) |>

  summarize(
    mean_pps = mean(price_per_sqft, na.rm = TRUE),
    median_pps = median(price_per_sqft, na.rm = TRUE),
    home_count = n()
  ) |>

  arrange(desc(mean_pps))

#### Reflection

Between the two, dplyr pipelines often feel clearer because the syntax reads more like a natural sentence, and the use of the pipe operator (|>) avoids the need to repeatedly reference the dataframe name. What is similar is the logical flow: both require explicit steps for filtering, creating variables, grouping, and summarizing. The difference lies in Python’s requirement for parentheses around boolean conditions and the use of lambda functions within .assign() to handle the temporary dataframe state during chaining, whereas R handles these implicitly.

Part C – Boolean Logic Debugging
1. Fixed Code:

In [8]:
df[(df["price"] > 250000) & (df["size"] > 1000)]

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,1500000,1280.741760,1.0,Suburb,Townhouse
1,100002,1500000,1406.283113,2.0,Uptown,SingleFamily
2,100003,1500000,4146.825713,6.0,Suburb,MultiFamily
3,100004,1500000,3946.599818,6.0,Suburb,SingleFamily
4,100005,1500000,1243.751760,1.0,Downtown,MultiFamily
...,...,...,...,...,...,...
595,100596,1500000,1443.241197,3.0,Midtown,Condo
596,100597,1500000,1083.909714,2.0,Suburb,Condo
597,100598,1500000,1600.126432,1.0,Suburb,SingleFamily
598,100599,1500000,1248.216637,1.0,Waterfront,Condo


2. Why the error occurs:

In Python, the bitwise operator & has higher precedence than comparison operators like >. Without parentheses, pandas attempts to evaluate 250000 & df["size"] first, which leads to a TypeError because you cannot perform a bitwise AND between an integer and a Series in that manner.

3. Rewrite using .query():

In [9]:
df.query("price > 250000 & size > 1000")

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,1500000,1280.741760,1.0,Suburb,Townhouse
1,100002,1500000,1406.283113,2.0,Uptown,SingleFamily
2,100003,1500000,4146.825713,6.0,Suburb,MultiFamily
3,100004,1500000,3946.599818,6.0,Suburb,SingleFamily
4,100005,1500000,1243.751760,1.0,Downtown,MultiFamily
...,...,...,...,...,...,...
595,100596,1500000,1443.241197,3.0,Midtown,Condo
596,100597,1500000,1083.909714,2.0,Suburb,Condo
597,100598,1500000,1600.126432,1.0,Suburb,SingleFamily
598,100599,1500000,1248.216637,1.0,Waterfront,Condo


Part D – Short Concept Questions

1. Why parentheses with &?      Due to operator precedence, Python evaluates bitwise operators (&, |) before comparisons (>, <). Parentheses ensure the logical comparisons are completed before the "AND" logic is applied.

2. Advantage of method chaining?       It keeps code clean and readable by showing the data's journey in a single block. It also prevents the creation of numerous temporary "intermediate" variables (like df2, df3) that clutter memory and the coding environment.

3. What do "price" and "mean" represent? "price" represents the column name in the dataframe you wish to aggregate. "mean" represents the statistical function (the mean) that you want to apply to that column.

4. Why does neighborhood appear on the left (index)? The .groupby() function makes the grouping variable the index of the resulting dataframe because that variable now uniquely identifies each row in the summary table.

Optional Extension

Summary table grouped by type:

In [13]:
type_summary = (
    df.groupby("type")
    .agg(
        mean_price = ("price", "mean"),
        median_price = ("price", "median"),
        listing_count = ("price", "count")
    )
    .sort_values("mean_price", ascending=False)
)
type_summary

,mean_price,median_price,listing_count
type,,,
Condo,1500000.0,1500000.0,183
MultiFamily,1500000.0,1500000.0,63
SingleFamily,1500000.0,1500000.0,235
Townhouse,1500000.0,1500000.0,119
